# Module 1: Deep Learning & Neural Networks for Student Question Classification
## AI Study Assistant - From Scratch

Welcome to **Module 1**! Is notebook mein hum ek custom **Deep Learning Multi-Layer Perceptron (MLP)** banayenge using **PyTorch** jo student questions ko classify karega in 3 domains:
1. **Python** (e.g., syntax, data structures, decorators, OOPs)
2. **AI (Artificial Intelligence)** (e.g., search algorithms, heuristics, expert systems, agents)
3. **ML (Machine Learning)** (e.g., supervised learning, loss functions, overfitting, regularization)

---

### Core Concepts in Simple Hinglish:

1. **Input Layer**: Yahan hamara numerical feature vector (TF-IDF vector) model ke andar enter karta hai. Agar vocabulary size 800 hai, toh input layer ke paas 800 nodes honge.
2. **Hidden Layer(s)**: Intermediate layers jo features ke complex patterns aur nonlinear combinations learn karte hain.
3. **Weights (W) & Biases (b)**:
   - **Weights**: Har input feature ki importance decide karte hain ($z = Wx + b$).
   - **Biases**: Line ya decision boundary ko shift karne ki flexibility dete hain.
4. **Activation Functions (ReLU)**:
   - Agar activation function na ho, toh chahe kitni bhi layers add kar lo, network sirf ek linear function bankar reh jayega.
   - **ReLU (Rectified Linear Unit)**: $f(x) = \max(0, x)$ - simple, computationally fast, aur vanishing gradient problem ko solve karta hai.
5. **Output Layer & Softmax**:
   - Hamare paas 3 classes hain (Python, AI, ML), isliye output layer ke 3 nodes honge (logits).
   - **Softmax** in logits ko valid probabilities mein convert karta hai jiska sum = 1 hota hai.
6. **Forward Propagation**:
   - Input vector forward direction mein travel karta hai through layers aur prediction generate hoti hai.
7. **Loss Function (Cross-Entropy Loss)**:
   - Actual ground truth aur predicted probabilities ke beech ka error measure karta hai.
8. **Backpropagation & Optimizer (Adam)**:
   - Calculus ke **Chain Rule** ka use karke output error se har weight aur bias ka gradient calculate hota hai.
   - Optimizer in weights ko update karta hai: $W_{new} = W_{old} - \eta \cdot \nabla W$.

## Step 1: Environment & Hardware Verification

In [ ]:
import sys
import os
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

## Step 2: Load and Explore the Labeled Dataset

In [ ]:
# Load dataset from data/raw
data_path = "../data/raw/student_questions.csv"
df = pd.read_csv(data_path)
print("Total Questions:", len(df))
print("\nClass Distribution:")
print(df["category"].value_counts())
df.head()

## Step 3: Text Preprocessing & TF-IDF Feature Extraction
Text data raw format me neural network me feed nahi kiya ja sakta. Isliye hum:
1. Punctuation aur extra spaces clean karte hain.
2. TF-IDF (Term Frequency - Inverse Document Frequency) se numerical vectors generate karte hain.

In [ ]:
sys.path.insert(0, "..")
from src.classification.dataset import TextPreprocessor, DataPipeline

pipeline = DataPipeline(max_features=800, batch_size=16, random_state=42)
data_bundle = pipeline.load_and_prepare(data_path)

print("Classes:", data_bundle["classes"])
print("Feature Dimension (Vocabulary Size):", data_bundle["feature_dim"])
print("Train Batches:", len(data_bundle["train_loader"]))
print("Val Batches:", len(data_bundle["val_loader"]))
print("Test Batches:", len(data_bundle["test_loader"]))

## Step 4: PyTorch Neural Network Architecture
Multi-Layer Perceptron (MLP) with Batch Normalization, ReLU non-linearity, and Dropout.

In [ ]:
from src.classification.model import QuestionClassifierMLP

model = QuestionClassifierMLP(
    input_dim=data_bundle["feature_dim"],
    hidden_dim1=128,
    hidden_dim2=64,
    num_classes=data_bundle["num_classes"],
    dropout_rate=0.3
).to(device)

print(model)
print("\nModel Information:", model.get_model_info())

## Step 5: Test Model Prediction with Confidence Scores
Hum trained weights aur artifacts (`models/question_classifier.pt`) ko load karke naye student questions classify karenge.

In [ ]:
from src.classification.predict import QuestionClassifier

classifier = QuestionClassifier(
    model_path="../models/question_classifier.pt",
    vectorizer_path="../models/tfidf_vectorizer.pkl",
    encoder_path="../models/label_encoder.pkl"
)

sample_questions = [
    "How does the GIL affect multithreading in Python?",
    "Can you explain the A* search algorithm and heuristic functions?",
    "What is the difference between L1 Lasso and L2 Ridge regularization?"
]

for q in sample_questions:
    res = classifier.predict(q)
    print(f"\nQuestion: {res['question']}")
    print(f"Predicted Subject : {res['category']} (Confidence: {res['confidence']*100:.2f}%)")
    print(f"All Probabilities : {res['probabilities']}")

## Step 6: Evaluation Metrics and Confusion Matrix
Inspect the saved model evaluation metrics.

In [ ]:
import json

with open("../models/model_metadata.json") as f:
    metadata = json.load(f)

print("Test Accuracy :", f"{metadata['test_accuracy']*100:.2f}%")
print("Weighted F1   :", f"{metadata['test_f1_weighted']:.4f}")
print("Macro F1      :", f"{metadata['test_f1_macro']:.4f}")
print("Test Loss     :", f"{metadata['test_loss']:.4f}")

# Plot Confusion Matrix
cm = np.array(metadata["confusion_matrix"])
classes = metadata["classes"]

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Question Classifier - Confusion Matrix")
plt.xlabel("Predicted Subject")
plt.ylabel("True Subject")
plt.tight_layout()
plt.show()